In [1]:
from z3 import *
import re
from copy import deepcopy
# Define System E Language
PointSort = DeclareSort('Point')
LineSort = DeclareSort('Line')
CircleSort = DeclareSort('Circle')
RightAngle = RealVal(90)
bet = Function('bet', PointSort, PointSort, PointSort, BoolSort())
onLine = Function('onLine', PointSort, LineSort, BoolSort())
onCircle = Function('onCircle', PointSort, CircleSort, BoolSort())
inside = Function('inside', PointSort, CircleSort, BoolSort())
center = Function('center', PointSort, CircleSort, BoolSort())
sameside = Function('sameside', PointSort, PointSort, LineSort, BoolSort())
# Intersection Predicates
interLL = Function('interLL', LineSort, LineSort, BoolSort())
interLC = Function('interLC', LineSort, CircleSort, BoolSort())
interCC = Function('interCC', CircleSort, CircleSort, BoolSort())
# --- Magnitudes / Metrics (Return RealSort) ---
Segment = Function('Segment', PointSort, PointSort, RealSort())
Angle = Function('Angle', PointSort, PointSort, PointSort, RealSort())
Area = Function('Area', PointSort, PointSort, PointSort, RealSort())
# --- Syntactic sugars ---
diffside = lambda a,b,L:  And(Not(onLine(a, L)), Not(onLine(b, L)), Not(sameside(a, b, L)))
outside = lambda a,α: And(Not(inside(a,α)), Not(onCircle(a,α)))
points = lambda *args: Or(Distinct(*args), True)   # A tautology for introducting points in premise
cong =  Function('cong', PointSort, PointSort, PointSort, PointSort, PointSort, PointSort, BoolSort())
coll = Function('coll', PointSort, PointSort, PointSort, BoolSort())
triangle = lambda a,b,c: And(Distinct(a,b,c), Not(coll(a,b,c)) )       # Non-degenerate triangle
insideTri = Function('insideTri', PointSort, PointSort, PointSort, PointSort, BoolSort())
sameside4 = Function('sameside4', PointSort, PointSort, PointSort, PointSort, BoolSort())

A, B, C, D, E,F,G,H,I,J,K,L,M,N,P,Q,R,S = Consts('A B C D E F G H I J K L M N P Q R S', LineSort)
a,b,c,d,e,f,g,h,i,j,k,l,m,n,o,p = Consts('a b c d e f g h i j k l m n o p', PointSort)
α,β,γ,δ,ε,ζ,η,θ,ι,κ,λ,μ,ω = Consts('α β γ δ ε ζ η θ ι κ λ μ ω', CircleSort)
## Const of Sort instance Initialiser
Point = lambda name: Const(name, PointSort)
Line = lambda name: Const(name, LineSort)
Circle = lambda name: Const(name, CircleSort)
## function objects
def line(a: PointSort, b: PointSort):
    newLine = Line(a.decl().name() + b.decl().name())
    return And( onLine(a,newLine), onLine(b,newLine) )


## ======== Axioms ==========
AXIOMS = { ## General Rules
'G1': ForAll([a,b,L,M], Implies( And(a != b, onLine(a,L), onLine(a,M), onLine(b,L), onLine(b,M)), L == M )), #兩點皆在線和線，則線線本同
'G2': ForAll([a,b,α,β], Implies( And(center(a,α), center(b,β)) , a == b )),  #點點忠圓則本同
'G3': ForAll([a,α], Implies( center(a,α) , inside(a,α) )),           #圓心入圓
'G4': ForAll([a,α], Implies( inside(a,α) , Not(onCircle(a,α)) )),    #圓內點不在圓
 ## Betweenness
'B1': ForAll([a,b,c], Implies( bet(a,b,c) , And( bet(c,b,a), Distinct(a,b,c), Not(bet(b,a,c)) )  )),#排則三點各異，首尾可換但首間不可換
'B2': ForAll([a,b,c,L], Implies( And(bet(a,b,c), onLine(a,L), onLine(b,L)), onLine(c,L)  )),   #線過排首兩點，則也過尾點
'B3': ForAll([a,b,c,L], Implies( And(bet(a,b,c), onLine(a,L), onLine(c,L)), onLine(b,L)  )),   #線過排首尾點，則也過間點
'B4': ForAll([a,b,c,d], Implies( And(bet(a,b,c), bet(a,d,b)), bet(a,d,c) )),    #排首兩之間點在首尾之間
'B5': ForAll([a,b,c,d], Implies( And(bet(a,b,c), bet(b,c,d)), bet(a,b,d) )),  #排尾兩是另排首兩，則間點在排首另排尾之間
'B6': ForAll([a,b,c,L], Implies( And(Distinct(a,b,c), onLine(a,L), onLine(b,L), onLine(c,L)),    #三點在線，必有一間點焉
                               And(AtLeast(bet(a,b,c),bet(b,a,c),bet(a,c,b), 1), AtMost(bet(a,b,c),bet(b,a,c),bet(a,c,b), 1)) )),
'B7': ForAll([a,b,c,d], Implies( And(bet(a,b,c), bet(a,b,d)), Not(bet(c,b,d)) )),   #兩排共首兩，則間點不在尾們之間
 ## Samesideness
'S1': ForAll([a,L], Implies( Not(onLine(a,L)), sameside(a,a,L) )),  #不在線點與己同邊
'S2': ForAll([a,b,L],  sameside(a,b,L) == sameside(b,a,L) ),   #點點同線邊可交換
'S3': ForAll([a,b,L], Implies( sameside(a,b,L), Not(onLine(a,L)) )),  #點點同線邊不在線
'S4': ForAll([a,b,c,L], Implies( And(sameside(a,b,L), sameside(a,c,L)), sameside(b,c,L) )),  #點點共點同線邊，則點點也同線邊
'S5': ForAll([a,b,c,L], Implies( And(Not(onLine(a,L)), Not(onLine(b,L)), Not(onLine(c,L)), Not(sameside(a,b,L)) ), 
                                 Xor(sameside(a,c,L),sameside(b,c,L)) )),   #點點點選線邊站，點點各一邊，則餘點必與點同邊
 ## Pasch axioms
'P1': ForAll([a,b,c,L], Implies( And(bet(a,b,c), sameside(a,c,L)), sameside(a,b,L)  )),   #排首尾同線邊，則首兩也同邊
'P2': ForAll([a,b,c,L], Implies( And(bet(a,b,c), onLine(a,L), Not(onLine(b,L))), sameside(b,c,L)  )),  #排首在線且間點不在線，則間尾同邊
'P3': ForAll([a,b,c,L], Implies( And(bet(a,b,c), onLine(b,L)), Not(sameside(a,c,L)) )),   #排間點在線，則首尾不同邊
'P4': ForAll([a,b,c,L,M], Implies( And(L!=M, onLine(b,L), onLine(b,M), onLine(a,M), onLine(c,M), Distinct(a,b,c), Not(sameside(a,c,L))), 
                                   bet(a,b,c) )),   #兩線有交點，另兩點皆在甲線且不同乙線邊，則交點為間點
## SKIPPED Triple incidence axioms FOR NOW
 ## Circles rules
'C1': ForAll([a,b,c,L,α], Implies( And(onLine(a,L),onLine(b,L),onLine(c,L), inside(a,α),onCircle(b,α),onCircle(c,α), b!=c), 
                                   bet(b,a,c) )),   #三點在線，其一入圓，餘兩點在圓，則入圓者在兩點之間
'C2': ForAll([a,b,c,α], Implies( And(Or(inside(a,α),onCircle(a,α)), Or(inside(b,α),onCircle(b,α)), bet(a,c,b)  ), 
                                   inside(c,α) )),   #排首尾皆在圓盤，則間點入圓
'C3': ForAll([a,b,c,α], Implies( And(Or(inside(a,α),onCircle(a,α)), Not(inside(c,α)), bet(a,c,b)  ), 
                                   And(Not(inside(b,α)), Not(onCircle(b,α))) )),   #排首在圓盤，間點不入圓，則尾出圓
'C4': ForAll([a,b,c,d,L,α,β], Implies( And(α!=β, c!=d, onCircle(c,α), onCircle(d,α), onCircle(c,β), onCircle(d,β), 
                                         center(a,α), center(b,β), onLine(a,L), onLine(b,L)   ), 
                                   Not(sameside(c,d,L)) )),    #兩圓交之兩點不同過各圓心之線邊
 ## Intersection rules
'I1': ForAll([a,b,L,M], Implies( And(Not(sameside(a,b,L)),Not(onLine(a,L)),Not(onLine(b,L)), onLine(a,M), onLine(b,M)  ), 
                                   interLL(L,M) )),   #隔開兩點之線必交過兩點之線
'I2': ForAll([a,b,L,α], Implies( And(Or(inside(a,α),onCircle(a,α)), Or(inside(b,α),onCircle(b,α)), 
                                     Not(sameside(a,b,L)),Not(onLine(a,L)),Not(onLine(b,L)) ), 
                                   interLC(L,α) )),   #隔開圓盤兩點之線交圓
'I3': ForAll([a,L,α], Implies( And(inside(a,α), onLine(a,L),   ), interLC(L,α) )),   #過圓內點之線交圓
'I4': ForAll([a,b,α,β], Implies( And(onCircle(a,α), Or(inside(b,α),onCircle(b,α)), inside(a,β), Not(Or(inside(b,β),onCircle(b,β))) ), 
                                   interCC(α,β) )),   #過圓內點並包含圓外點之另圓交圓
'I5': ForAll([a,b,α,β], Implies( And(onCircle(a,α), inside(b,α), inside(a,β), onCircle(b,β) ), 
                                   interCC(α,β) )),   #兩圓互過對方內點則相交
 ## Metric rules
'M1': ForAll([a,b], (Segment(a,b) == RealVal(0)) == (a == b) ),   #段長為零即端本同
'M2': ForAll([a,b], (Segment(a,b) >= RealVal(0)) ),  #段長非負
'M3': ForAll([a,b], Segment(a,b) == Segment(b,a) ),  #段端可互換
'M4': ForAll([a,b,c], Implies(And(a!=b, b!=c), Angle(a,b,c)==Angle(c,b,a)) ),  #角端不同頂，角端可互換
'M5': ForAll([a,b,c], And(RealVal(0) <= Angle(a,b,c), Angle(a,b,c) <= RealVal(180)) ),  #角度介乎零至一八零
'M6': ForAll([a,b], Area(a,a,b) == 0 ),  #兩角形面積為零
'M7': ForAll([a,b,c], Area(a,b,c) >= 0 ),   #面積非負
'M8': ForAll([a,b,c], And(Area(a,b,c) == Area(c,a,b), Area(c,a,b) == Area(a,c,b)) ),  #置換三角形頂點不變面積
'M9': ForAll([a,b,c,d,e,f], Implies(And(Segment(a,b)==Segment(d,e),Segment(b,c)==Segment(e,f), Segment(c,a)==Segment(f,d),
                                 Angle(a,b,c)==Angle(d,e,f), Angle(b,c,a)==Angle(e,f,d), Angle(c,a,b)==Angle(f,d,e) ),
                             Area(a,b,c) == Area(d,e,f) )),    #兩三角形對應邊角相等，則面積相等
 ## Diagram-segment transfer
'DS1': ForAll([a,b,c], Implies(bet(a,b,c), Segment(a,b) + Segment(b,c) == Segment(a,c) )),   #排首尾段長是首間間尾段長之和
'DS2': ForAll([a,b,c,α,β], Implies(And(center(a, α), center(a, β), onCircle(b, α), onCircle(c, β), Segment(a, b) == Segment(a, c)), 
                         α == β)),   #半徑相等的同心圓本同
'DS3': ForAll([a,b,c,α], Implies(And(center(a, α), onCircle(b, α)),   #惟點在圓時點心段長等於半徑
                                 (Segment(a, c) == Segment(a, b)) == onCircle(c, α) )),
'DS4': ForAll([a,b,c,α], Implies(And(center(a, α), onCircle(b, α)),   #惟點入圓時點心段長小於半徑
                                  (Segment(a, c) < Segment(a, b)) == inside(c, α) )),
 ## Diagram-angle transfer
'DA1': ForAll([a,b,c,L], Implies(And(a != b, a != c, onLine(a, L), onLine(b, L)),#三點成角，頂和一端在線，則惟另端在線且頂非於兩端之間時，角度為零
                             And(onLine(c, L), Not(bet(b, a, c))) == (Angle(b, a, c) == RealVal(0)) )),
'DA2': ForAll([a,b,c,d,L,M],   #兩線各過角端交於角頂，則惟有另點丁在角裏時，角度為端頂丁和丁頂另端之和
                 Implies(And(onLine(a, L), onLine(a, M), onLine(b, L), onLine(c, M), 
                             a != b, a != c, Not(onLine(d, L)), Not(onLine(d, M)), L != M),
                         (Angle(b, a, c) == Angle(b, a, d) + Angle(d, a, c)) == And(sameside(b, d, M), sameside(c, d, L)))),
'DA3': ForAll([a, b, c, d, L],   #兩鄰角相等惟皆直角時
                 Implies(And(onLine(a, L), onLine(b, L), bet(a, c, b), Not(onLine(d, L))),
                         (Angle(a, c, d) == Angle(d, c, b)) == (Angle(a, c, d) == RealVal(90)))),
'DA4': ForAll([a, b, c, e, f],    #兩角共頂點，角邊各共線，各頂和端同向，則兩角度相等
                 Implies(And(coll(a,b,e), coll(a,c,f), b != a, e != a, c != a, f != a, Not(bet(b, a, e)), Not(bet(c, a, f))),
                         Angle(b, a, c) == Angle(e, a, f))),
'DA5': ForAll([a, b, c, d, e, L, M, N],   #同旁內角和小於平角，則兩被截線相交於角端之側
                 Implies(And(onLine(a, L), onLine(b, L), onLine(b, M), onLine(c, M), onLine(c, N), onLine(d, N), 
                             b != c, sameside(a, d, M), (Angle(a, b, c) + Angle(b, c, d)) < RealVal(180)),
                         And(interLL(L, N), Implies(And(onLine(e, L), onLine(e, N)), sameside(e, a, M))) )),
 ## Diagram-area transfer
'D1': ForAll([a, b, c, L],  # 兩點在線，與另點成形，惟該點也在線時面積為零
        Implies(And(onLine(a, L), onLine(b, L), a != b), (Area(a, b, c) == RealVal(0)) == onLine(c, L))),
'D2': ForAll([a, b, c, d, L],  #三點在線，另點不在，則惟三點成排時，首尾另面積為首間另與另間尾之和
                    Implies(And(onLine(a, L), onLine(b, L), onLine(c, L), Distinct(a, b, c), Not(onLine(d, L))),
                         bet(a, c, b) == (Area(a, c, d) + Area(d, c, b) == Area(a, d, b)))),
 ## Superimposition rule (SAS axiom)
'E4': ForAll([a,b,c,d,e,f], Implies(And(Segment(a,b)==Segment(d,e),Segment(b,c)==Segment(e,f), Angle(a,b,c)==Angle(d,e,f)  ),
                                 cong(a,b,c,d,e,f) )),    #兩三角形兩邊角和夾角相應，則全等
 ## Triangle Congruence
'CG': ForAll([a,b,c,d,e,f], cong(a,b,c,d,e,f) == And(Segment(a,b)==Segment(d,e),Segment(b,c)==Segment(e,f), Segment(c,a)==Segment(f,d),
                 Angle(a,b,c)==Angle(d,e,f), Angle(b,c,a)==Angle(e,f,d), Angle(c,a,b)==Angle(f,d,e))  )  # Triangle Congreunce def

}
## ======= CONSTRUCTION RULES ======== 
CONSTRUCTION_RULES = {  ## === Points ===     each rule consists of [ args => premises, args => conclusions ]
# CP1: 有點同另點  ∃ a 使 a = b
'CP1': [ lambda b: BoolVal(True),     
        lambda a, b: a == b ],
# CP2: 線上有點  ∃ a 使 a on L
'CP2': [ lambda L: BoolVal(True),     
        lambda a, L: And(onLine(a, L)) ],
# CP3: 兩點之間有點  b c on L, b ≠ c ⊢ ∃a 使 a on L ∧ bet(b,a,c)
'CP3': [ lambda b, c, L: And(onLine(b, L), onLine(c, L), b != c ),
         lambda a, b, c, L: And(onLine(a, L), bet(b, a, c)) ],
# CP4: 兩點之後有點  b, c on L, b ≠ c ⊢ a on L ∧ bet(b,c,a)
'CP4': [ lambda b, c, L: And(onLine(b, L), onLine(c, L), b != c),
         lambda a, b, c, L: And(onLine(a, L), bet(b, c, a)) ],
# CP5: 有點同邊不在線點 (¬(b on L) ⊢ a, b sameside L)
'CP5': [ lambda b, L: Not(onLine(b, L)),
         lambda a, b, L: sameside(a, b, L) ],
# CP6: 有點異邊不在線點 (¬(b on L) ⊢ a, b diffside L)
'CP6': [ lambda b, L: Not(onLine(b, L)),
         lambda a, b, L: diffside(a, b, L) ],
# CP7: 圓上有點  ∃a 使 a on α
'CP7': [ lambda α: BoolVal(True),
         lambda a, α: onCircle(a, α) ],
# CP8: 圓內有點  ∃a 使 a inside α 
'CP8': [ lambda α: BoolVal(True),
         lambda a, α: inside(a, α) ],
# CP9: 圓外有點  ∃a 使 a outside α
'CP9': [ lambda α: BoolVal(True),
         lambda a, α: And(outside(a, α)) ],
## ===== Lines and circles =====
# LC1: 兩點有一線過 (a ≠ b ⊢ ∃L 使 a on L ∧ b on L)
'LC1': [ lambda a, b: a != b,
         lambda L, a, b: And(onLine(a, L), onLine(b, L)) ],
# LC2: 有圓中心點過另點 (a ≠ b ⊢ ∃ α 使 a center α ∧ b on α)
'LC2': [ lambda a, b: a != b,
         lambda α, a, b: And(center(a, α), onCircle(b, α)) ],
## ===== Intersections =====
# CI1: 線交線有點  L inter M ⊢ ∃a 使 a on L ∧ a on M
'CI1': [ lambda L, M: interLL(L, M),
         lambda a, L, M: And(onLine(a, L), onLine(a, M)) ],
# CI2: 圓交線有點  α inter L ⊢ ∃a 使 a on α ∧ a on L
'CI2': [ lambda α, L: interLC(L, α),
         lambda a, α, L: And(onCircle(a, α), onLine(a, L)) ],
# CI3: 圓交線有兩點  α inter L ⊢ ∃a b 使a b on α ∧ a b on L ∧ a ≠ b
'CI3': [ lambda α, L: interLC(L, α),
         lambda a, b, α, L: And(onCircle(a, α), onLine(a, L), onCircle(b, α), onLine(b, L), a != b) ],
# CI4: 線過圓內外兩點，則兩點之間有交點  b inside α, b on L, c outside α, c on L  ⊢ ∃a 使a on α ∧ a on L ∧bet(b,a,c)
'CI4': [ lambda b, c, L, α: And(inside(b, α), onLine(b, L), outside(c, α), onLine(c, L)),
         lambda a, b, c, L, α: And(onCircle(a, α), onLine(a, L), bet(b, a, c)) ],
# CI5: 線過點和圓內另點，則兩點之後有交點  b inside α, b on L, c ≠ b, c on L ⊢ ∃a 使a on α ∧ a on L ∧ bet(a,b,c)
'CI5': [ lambda b, c, L, α: And(inside(b, α), onLine(b, L), c != b, onLine(c, L)),
         lambda a, b, c, L, α: And(onCircle(a, α), onLine(a, L), bet(a, b, c)) ],
# CI6: 圓交圓有點  α inter β ⊢ ∃a 使 a on α ∧ a on β
'CI6': [ lambda α, β: interCC(α, β),
         lambda a, α, β: And(onCircle(a, α), onCircle(a, β)) ],
# CI7: 圓交圓有兩點  α inter β ⊢ ∃a b 使 a on α β ∧ b on α β ∧ a ≠ b
'CI7': [ lambda α, β: interCC(α, β),
         lambda a, b, α, β: And(onCircle(a, α), onCircle(a, β), onCircle(b, α), onCircle(b, β), a != b) ],
# CI8: 圓交圓有點在過各心之線的點同邊  α inter β, c center α, d center β. c d on L, ¬(b on L) ⊢ ∃a 使a on α ∧ a on β ∧ a b sameside L
'CI8': [ lambda b, c, d, L, α, β: And(interCC(α, β), center(c, α), center(d, β), onLine(c, L), onLine(d, L), Not(onLine(b, L))),
         lambda a, b, c, d, L, α, β: And(onCircle(a, α), onCircle(a, β), sameside(a, b, L)) ],
# CI9: 圓交圓有點在過各心之線的點異邊  α inter β, c center α, d center β, c d on L, ¬(b on L) ⊢ ∃a 使a on α ∧ a on β ∧ a b diffside L
'CI9': [ lambda b, c, d, L, α, β: And(interCC(α, β), center(c, α), center(d, β), onLine(c, L), onLine(d, L), Not(onLine(b, L))),
         lambda a, b, c, d, L, α, β: And(onCircle(a, α), onCircle(a, β), diffside(a, b, L)) ],
## EXTRAS (for convenience)
# SS: sameside4 to sameside
'SS': [ lambda d,a,b,c: sameside4(d,a,b,c),
         lambda L,d,a,b,c: And( onLine(b,L),onLine(c,L), sameside(d,a,L) ) ],
}
## ======= DEFAULT AXIOMS (no need to supply reason) =======
DEFAULTS = {  
'B1part': ForAll([a,b,c], Implies( bet(a,b,c) , And( bet(c,b,a), Distinct(a,b,c), Not(bet(b,a,c)) )  )),
'S2': AXIOMS['S2'],
'M0': ForAll([a], Segment(a,a) == RealVal(0)  ),    # M0: Segment(a,a) = 0      // 單點無長
'M2': AXIOMS['M2'],    # M2: Segment(a,b) ≥ 0      // 段長非負
'M3': AXIOMS['M3'],    # M3: Segment(a,b) = Segment(b,a)    // 段端可互換
'M4': AXIOMS['M4'],    # M4: a ≠ b, b ≠ c ⊢ ∠(a,b,c) = ∠(c,b,a)   // 角端不同頂，角端可互換
'M7': AXIOMS['M7'],    # M7: Area(a,b,c) ≥ 0   // 面積非負
'L0': ForAll([a,b,c], coll(a,b,c) == Or( bet(a,b,c), bet(b,c,a), bet(c,a,b), a==b, b==c, a==c )),   # Collinearity def
'L1': ForAll([a,b,c], Implies( bet(a,b,c), coll(a,b,c) )),  # Collinearity (for convenience) #排則共線
'L2': ForAll([a,b,c],  coll(a,b,c) == coll(c,b,a) ),   # Terminals can swap
'L3': ForAll([a,b,c],  coll(a,b,c) == coll(b,a,c) ),   # First two can swap
'L4': ForAll([a,b,c,L], Implies( And(onLine(a,L),onLine(b,L),onLine(c,L)), coll(a,b,c) )),   # Collinearity def
'CG': AXIOMS['CG'],

'SS': ForAll([a,b,c,d,L], Implies( And(onLine(b,L),onLine(c,L), sameside(a,d,L) ), sameside4(a,d,b,c) )  ),
'IT': ForAll([a,b,c,d], insideTri(d,a,b,c) == And( sameside4(d,a,b,c), sameside4(d,b,a,c), sameside4(d,c,a,b)   )),   # inside triangle def
}


In [2]:
class Proof:
    def __init__(self, premises, goal, solver: Solver = None):
        if solver is None:
            solver = Solver(); solver.set("timeout", 2000)
        self.solver = solver
        self.axioms = AXIOMS
        self.z3_vars = {}    # Store Z3 variables, indexed by str
        self.z3_vars_hist = []    # For cases variables revertion
        self.context: [Bool] = [ None ]    # Stores premises and any valid formulas proved
        self.context_hist = []    # For cases context revertion
        self.goal = 'undefined'
        self.premises = 'undefined'
        self.premise_vars = set()
        self.goal_vars = set()
        self.goalonly_vars = set()   # = goal_vars - premise_vars
        self.loadContextAndGoal(premises, goal)
        ## For storing tactics info
        self.cases_layer = 0   # Each case split will +1 until resolved
        self.cases_stack: [[ExprRef]] = []
        self.current_case = None
        self.current_case_nth = None
        self.current_or_expr = None

    def loadContextAndGoal(self, premises:list[ExprRef], goal:ExprRef):   # Called at init
        self.premises= deepcopy(premises)
        self.context = premises
        self.solver.add(premises)
        self.goal = goal
        for i,t in enumerate(premises):    # Print premises and conclusion for view
            print(f"{i+1}. {t}", end='\n' if i< len(premises)-1 else '')
        print(f'    / {self.goal}')
        print('-'*20)
        
        # 3. Populate self.z3_vars directly using the extracted Z3 objects
        for premise in premises:
            premise_vars = self.collect_vars(premise)
            self.premise_vars.update(premise_vars)
            for var_obj in premise_vars:
                name = var_obj.decl().name()
                if name not in self.z3_vars:
                    self.z3_vars[name] = var_obj
        self.goal_vars.update(self.collect_vars(goal))
        self.goalonly_vars = self.goal_vars - self.premise_vars
        # 4. Sorted premise_vars and conclusion_vars
        self.sorted_premise_vars = sorted(list(self.premise_vars), key=lambda t: (
            0 if 'a' <= t.decl().name()[0] <= 'z' else 
            1 if 'A' <= t.decl().name()[0] <= 'Z' else 2, 
            t.decl().name()  ))
        self.sorted_goalonly_vars = sorted(list(self.goalonly_vars), key=lambda t: (
            0 if 'a' <= t.decl().name()[0] <= 'z' else 
            1 if 'A' <= t.decl().name()[0] <= 'Z' else 2, 
            t.decl().name()  ))

        # 5. (Extra) Add some default axioms to make it easier for the user
        for rule in DEFAULTS.values():
            self.solver.add(rule)
        return

    # Helper: Native AST Variable Harvester (No Regex, No eval!)
    def collect_vars(self, expr):
        if len(expr.children()) == 0:   # If it has no children, it's a leaf node
            if expr.decl().name() not in ['True', 'False']:  # Skip boolean literals like True/False
                return {expr}
            return set()
        found = set()
        for child in expr.children():  # Otherwise, recursively gather variables from all sub-expressions
            found.update(self.collect_vars(child))
        return found
    # Helper: add expression to both context and solver
    def add(self, expr):
        self.solver.add(expr); self.context.append(expr)
        
    # Use rule to construct a new variable from existing objs
    def construct(self, rule:str, new: SortRef, objs: [SortRef], debug=True):  
        targets = new if isinstance(new, (list, tuple)) else [new]
        var_name = new.decl().name()
        # 1. Check if ANY of the target variable names are already in use
        for t in targets:
            var_name = t.decl().name()
            if var_name in self.z3_vars:
                print(f"--> #{len(self.context)+1} construct{rule, new, objs}: Variable '{var_name}' is already in use!")
                return
        # 2. Extract rule logic formulas
        try:
            premises = CONSTRUCTION_RULES[rule][0](*objs)
            conclusion = CONSTRUCTION_RULES[rule][1](*targets, *objs)
        except KeyError:
            print(f"--> #{len(self.context)+1} construct{rule, new, objs}: Invalid construction rule!")
            return
        except TypeError as e:
            print(f"--> #{len(self.context)+1} construct{rule, new, objs}: Argument count mismatch: {e}")
            return
        except Exception as e:
            print(f"--> #{len(self.context)+1} construct{rule, new, objs}: {e}!")
            return
        if conclusion in self.context:
            print(f"--> #{len(self.context)+1} construct{rule, new, objs}: You have constructed {conclusion} before. Try again.")
            return
        # 3. Check if all the rule's premises can be inferred from self.context
        self.solver.push()    # Make checkpoint
        self.solver.add( Not(premises) )
        if self.solver.check() == sat:   # Can't construct as premises not met
            print(f"--> #{len(self.context)+1} construct{rule,new,objs}:", end=' ')
            print(f"Sorry, can't construct {new} as {premises} are not met.", end='\n' if not debug else ' In particular:\n')
            if debug and premises.decl().name() == 'and':
                for prem in premises.children():
                    self.have( prem, debug=True )
            self.solver.pop(); return
        self.solver.pop()
        # 4. Success State Commitments
        self.add(conclusion)
        # Register the newly created variables to prevent future collisions
        for t in targets:
            self.z3_vars[t.decl().name()] = t
        print(f"{len(self.context)}. Constructed {new} where {conclusion} \t[{rule}]")
        return

    def have(self, fmla: ExprRef, rule: str=None, debug=False):    # Infer step
        self.solver.push()    # Make checkpoint
        if rule == 'sorry':   # CHEAT rule to allow to use EVERY AXIOM (but not constructions)
            for rue in AXIOMS:
                self.solver.add(AXIOMS[rue])
        elif rule is not None and rule not in DEFAULTS:
            try:
                self.solver.add( AXIOMS[rule] )   # Only allow this axiom in solver
            except KeyError:
                print(f"--> #{len(self.context)+1} have({fmla}, {rule}):", end=' ')
                print("Invalid reason!") 
            except Exception as e:
                print(f"--> #{len(self.context)+1} have({fmla}, {rule}): {e}!")
                return
                self.solver.pop(); return
        if fmla in self.context:
            print(f"--> #{len(self.context)+1} have({fmla}, {rule}):", end=' ')
            print(f"You have proved {fmla} before. Try again.")
            self.solver.pop(); return
        self.solver.add( Not(fmla) )
        result = self.solver.check()

        if result == sat or result == unknown:    # Can't infer
            print(f"--> #{len(self.context)+1} have({fmla}, {rule}):", end=' ')
            print(f"Sorry, can't infer {fmla} from {rule}.", end='\n')
            self.solver.pop();   # Remove the axiom used from solver 
            return
            # if debug and AXIOMS[rule].children() .decl().name() == 'and':
            #     print(' In particular:')
            #     for prem in fmla.children():
            #         self.have( prem, debug=True)
            # else: print('')
        # Successful deduction step
        print(f"{len(self.context)+1}. {fmla}\t[{rule}]" if not debug else f"SATISFIED. {fmla}\t[{rule}]")
        self.solver.pop();   # Remove the axiom used from solver 
        if not debug: 
            self.add( fmla ); return

    def qed(self, prop_name: str = None):
        # 0. Check if solver is inside a case split
        if self.cases_layer > 0:
            print(f"❌ Still have {self.cases_layer} layer of unresolved cases.")
            return
        # 1. Check if any formula in self.context matches the goal
        for fmla in self.context:
            if fmla.eq(self.goal):
                print(f"🔥 Success! You proved {self.goal} is true.")
                # Make sure we have a single base premise expression to work with
                if isinstance(self.premises, list):
                    base_premise = self.premises[0] if len(self.premises) == 1 else And(self.premises)
                else:
                    base_premise = self.premises
    
                # Snapshot references to our original variables
                orig_inputs = list(self.sorted_premise_vars)
                orig_outputs = list(self.sorted_goalonly_vars)
                orig_goal = self.goal
    
                # 2. Dynamic Closure for the Rule Premise
                def premise_closure(*runtime_inputs):
                    # Maps original proof inputs -> whatever new variables the user passes in
                    sub_pairs = list(zip(orig_inputs, runtime_inputs))
                    return substitute(base_premise, *sub_pairs)
    
                # 3. Dynamic Closure for the Rule Conclusion
                def goal_closure(*runtime_args):
                    # runtime_args matches the execution signature: (*outputs, *inputs)
                    runtime_outputs = runtime_args[:len(orig_outputs)]
                    runtime_inputs = runtime_args[len(orig_outputs):]
                    
                    # Zip old variable references to new ones across the entire scope
                    old_vars = orig_outputs + orig_inputs
                    new_vars = list(runtime_outputs) + list(runtime_inputs)
                    sub_pairs = list(zip(old_vars, new_vars))
                    
                    return substitute(orig_goal, *sub_pairs)
    
                # 4. Save the executable closures directly into AXIOMS/CONSTRUCTION_RULES
                if len(self.goalonly_vars) > 0:
                    # CASE A: Standard Construction Rule (Creates new points/lines/circles)
                    if prop_name not in CONSTRUCTION_RULES:
                        CONSTRUCTION_RULES[prop_name] = [premise_closure, goal_closure]
                        
                        pvars_str = ', '.join([t.decl().name() for t in self.sorted_premise_vars])
                        gvars_str = ', '.join([t.decl().name() for t in self.sorted_goalonly_vars])
                        print(f"💾 Added Construction Rule {prop_name}: [lambda {pvars_str}: {base_premise}, "
                              f"lambda {gvars_str}, {pvars_str}: {orig_goal}] into CONSTRUCTION RULES.")
                else:
                    # CASE B: Pure Deduction Theorem (No new variables, e.g., Prop 5)
                    # Build a universally quantified Z3 implication
                    if len(orig_inputs) > 0:
                        theorem_formula = ForAll(orig_inputs, Implies(base_premise, orig_goal))
                    else:
                        theorem_formula = Implies(base_premise, orig_goal)
                    
                    # Store it in a global dictionary for theorems/axioms
                    # (Make sure to define THEOREMS = {} globally alongside CONSTRUCTION_RULES)
                    AXIOMS[prop_name] = theorem_formula
                    
                    print(f"💾 Registered {prop_name} as a Universal Theorem in AXIOMS!")
                    print(f"   Form: {theorem_formula}")
            
                return True
    
        print(f"❌ You haven't proved {self.goal} yet.")
        return False
    ## ===== Proof Tactics =====
    # Helpers for saving self.context state
    def push(self):
        self.solver.push()                            # solver snapshot
        self.context_hist.append(list(self.context))  # context snapshot
        self.z3_vars_hist.append(dict(self.z3_vars))  # variables snapshot
        #print('scv(pushed):', self.solver, '|', self.context_hist, '|', self.z3_vars_hist)
    def pop(self):
        if not self.context_hist:
            raise ValueError("No checkpoint to pop!")
        self.solver.pop()                       # solver rolls back
        self.context = self.context_hist.pop()  # context rolls back to the exact same moment
        self.z3_vars = self.z3_vars_hist.pop()  # variables rolls back
        #print('scv(popped):', self.solver, '|', self.context_hist, '|', self.z3_vars_hist)
    # By cases (Or-elim)
    def cases(self, or_expr: ExprRef):
        self.cases_layer += 1
        if or_expr.decl().name() != 'or':
            print(f"--> #{len(self.context)+1} case({or_expr}): Can't case split on non-Or expression.")
            return
        if not any(or_expr.eq(c) for c in self.context):
            print(f"--> #{len(self.context)+1} case({or_expr}): This Or expression is not in the context.")
            return
        self.current_or_expr = or_expr
        self.push()   # checkpoint trioth solver and context and z3_vars
        children = list(reversed(or_expr.children()))
        self.cases_stack.append(children )
        first_case = children.pop()
        self.current_case = first_case; self.current_cases_nth = 1
        self.add(first_case)   # Assume the first case and try to reach the goal
        print(f"{len(self.context)}.{self.current_cases_nth}. Assuming case {first_case}. Try to reach the goal.")
    # Close the case if reached the goal
    def close(self):
        if self.cases_layer == 0:
            print(f"--> #{len(self.context)+1} case({or_expr}): There is no cases to close!"); return
        for fmla in self.context:
            if fmla.eq(self.goal):
                print(f"📁 Case closed! {self.current_case}'s goal reached.")
                self.pop()   # Revert to checkpoint
                if len(self.cases_stack[-1]) > 0:  # Move on to next case
                    next_case = self.cases_stack[-1].pop()
                    self.push()    # checkpoint
                    self.current_case = next_case
                    self.add(next_case); self.current_cases_nth += 1
                    print(f"{len(self.context)}.{self.current_cases_nth}. Assuming case {next_case}. Try to reach the goal.")
                    return
                # All cases have been proved
                self.cases_layer -= 1
                self.add(self.goal)
                print(f"All cases closed! {self.goal} have been added to context.")
                return
        print(f"--> #{len(self.context)+1} close(): Sorry, cannot reach goal {self.goal} from assumption {self.current_case}.")
        return
        

In [6]:
''' I want to make line(a,b) a Line object with the property onLine(a,line(a,b)) and onLine(b,line(a,b)) '''

' I want to make line(a,b) a Line object with the property onLine(a,line(a,b)) and onLine(b,line(a,b)) '

In [5]:
# Proposition 7: Given triangle a b c, if c d sameside ab, ac = ad, cb = db, prove c = d
ab = Line('ab'); bc = Line('bc'); ac = Line('ac')
pc = Proof( [triangle(a,b,c), And(onLine(a,ab), onLine(b,ab) ), sameside(c,d,ab), 
           Segment(a,c)==Segment(a,d), Segment(c,b)==Segment(d,b) ], c == d )
pc.have( Or( c!=d, c==d ) )
pc.cases( Or( c!=d, c==d ) )
pc.have( Not(onLine(d,ab)), 'S3' )
pc.have( d!=a )
pc.have( Distinct(a,c,d) )
pc.have( Angle(a,c,d) == Angle(a,d,c), 'E5' )
pc.have( Distinct(b,c,d) )
pc.have( Angle(b,c,d) == Angle(b,d,c), 'E5' )
pc.have( Or( insideTri(d,a,b,c), Not(insideTri(d,a,b,c)) ) )
pc.cases( Or( insideTri(d,a,b,c), Not(insideTri(d,a,b,c)) ) )
pc.have( sameside(d,a,bc), 'SS' )

pc.have( Angle(a,d,c) > Angle(d,c,b), 'sorry' )

pc.have( Not(onLine(c,ab)), 'S3' )

1. And(Distinct(a, b, c), Not(coll(a, b, c)))
2. And(onLine(a, ab), onLine(b, ab))
3. sameside(c, d, ab)
4. Segment(a, c) == Segment(a, d)
5. Segment(c, b) == Segment(d, b)    / c == d
--------------------
6. Or(c != d, c == d)	[None]
7.1. Assuming case c != d. Try to reach the goal.
8. Not(onLine(d, ab))	[S3]
9. d != a	[None]
10. Distinct(a, c, d)	[None]
--> #11 have(Angle(a, c, d) == Angle(a, d, c), E5): Invalid reason!
--> #11 have(Angle(a, c, d) == Angle(a, d, c), E5): Sorry, can't infer Angle(a, c, d) == Angle(a, d, c) from E5.
11. Distinct(b, c, d)	[None]
--> #12 have(Angle(b, c, d) == Angle(b, d, c), E5): Invalid reason!
--> #12 have(Angle(b, c, d) == Angle(b, d, c), E5): Sorry, can't infer Angle(b, c, d) == Angle(b, d, c) from E5.
12. Or(insideTri(d, a, b, c), Not(insideTri(d, a, b, c)))	[None]
13.1. Assuming case insideTri(d, a, b, c). Try to reach the goal.
--> #14 have(sameside(d, a, bc), SS): Sorry, can't infer sameside(d, a, bc) from SS.
--> #14 have(Angle(a, d, c) > Angle

In [14]:
B.sort() == PointSort

False

In [21]:
Const('ab', LineSort) is Const('ab', LineSort) 

False

In [25]:
And([True,True])

And(True, True)

In [29]:
And([True]).children()

[True]